# Storm Claims Adjuster Allocation Optimization

This notebook demonstrates a small mixed-integer optimization model for assigning synthetic insurance claims to field adjusters after a storm. The objective is to minimize total travel distance while respecting adjuster capacity and a total-distance constraint.

> **Portfolio note:** The data are synthetic and the geography is simplified to representative state coordinates.

## 1. Setup

Install the dependencies from `requirements.txt` before running the notebook.

In [1]:
from IPython.display import HTML, display
from ipywidgets import IntSlider, interact

from storm_claims_optimization import (
    calculate_kpis,
    create_map,
    generate_scenario,
    solve_assignment,
)


## 2. Interactive optimization

Change the sliders to generate a scenario and re-solve the assignment model. The distance budget is expressed in **kilometres**, not dollars.

In [2]:
def optimize_and_plot(storm_severity=5, num_adjusters=5, distance_budget_km=50_000):
    """Generate a scenario, solve it, display KPIs, and render the map."""
    claims, adjusters, storm_center = generate_scenario(
        storm_severity=storm_severity,
        num_adjusters=num_adjusters,
    )

    try:
        result = solve_assignment(
            claims=claims,
            adjusters=adjusters,
            max_total_distance_km=distance_budget_km,
        )
    except ValueError as exc:
        display(HTML(f"<b>Scenario infeasible:</b> {exc}"))
        return

    kpis = calculate_kpis(result)
    summary = f"""
    <h4>KPI Summary</h4>
    <ul>
      <li><b>Storm center:</b> {storm_center}</li>
      <li><b>Total assignment distance:</b> {kpis['total_distance_km']:,.0f} km</li>
      <li><b>Average distance per claim:</b> {kpis['avg_distance_km']:,.0f} km</li>
      <li><b>Assignments within 500 km SLA:</b> {kpis['sla_rate']:.1%}</li>
      <li><b>Average adjuster utilization:</b> {kpis['avg_utilization']:.1%}</li>
    </ul>
    """
    display(HTML(summary))
    display(create_map(result))


interact(
    optimize_and_plot,
    storm_severity=IntSlider(value=5, min=1, max=10, step=1, description="Severity"),
    num_adjusters=IntSlider(value=5, min=1, max=20, step=1, description="Adjusters"),
    distance_budget_km=IntSlider(
        value=50_000,
        min=10_000,
        max=200_000,
        step=5_000,
        description="Distance cap",
    ),
);


interactive(children=(IntSlider(value=5, description='Severity', max=10, min=1), IntSlider(value=5, descriptio…

## 3. Interpretation

The model demonstrates a capacitated assignment problem: each claim is assigned once, each adjuster has a workload limit, and the optimizer minimizes aggregate travel distance. Green assignment lines meet the illustrative 500 km SLA threshold; red lines exceed it.

Because the scenario uses synthetic state-level coordinates and geodesic distance, the results should be interpreted as a modeling demonstration rather than operational recommendations.